# BART-LARGE Training - Grammar Correction + Educational Feedback
## Using facebook/bart-large for better text generation


In [1]:
# Install required packages
!pip install transformers datasets accelerate evaluate
!pip install rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=f7bc3b4ef96f731a3842044ea9d3297631dd1264809fbdc2ea1e98538fe20438
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [2]:
# Import libraries
import json
import torch
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    TrainingArguments, Trainer,
    DataCollatorForSeq2Seq
)
from datasets import Dataset, DatasetDict
from google.colab import files
import numpy as np
from sklearn.model_selection import train_test_split
import evaluate

In [3]:
# Upload your JSONL file
print("Upload your cbc_dataset.jsonl file:")
uploaded = files.upload()
jsonl_file = list(uploaded.keys())[0]
print(f"Uploaded: {jsonl_file}")


Upload your cbc_dataset.jsonl file:


Saving cbc_dataset.jsonl to cbc_dataset.jsonl
Uploaded: cbc_dataset.jsonl


In [4]:
# Load BART-LARGE model and tokenizer
model_name = "facebook/bart-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print(f"Model loaded: {model_name}")
print(f"Model parameters: {model.num_parameters():,}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size:,}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Model loaded: facebook/bart-large
Model parameters: 406,291,456
Tokenizer vocab size: 50,265


In [5]:
# Check GPU and setup device
print("=== GPU SETUP ===")
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    device = torch.device("cuda")
else:
    print("GPU not available, using CPU")
    device = torch.device("cpu")

print(f"Using device: {device}")

# Move model to GPU
model = model.to(device)
print(f"Model moved to: {next(model.parameters()).device}")

=== GPU SETUP ===
GPU available: NVIDIA A100-SXM4-40GB
GPU memory: 39.6 GB
Using device: cuda


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Model moved to: cuda:0


In [6]:
# BART-LARGE DUAL TASK APPROACH
def load_data_bart_dual_task(filename):
    correction_data = []
    feedback_data = []

    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                item = json.loads(line)

                # Parse the output to extract correction and feedback
                output = item['output']
                if 'CORRECTION:' in output and 'FEEDBACK:' in output:
                    correction = output.split('CORRECTION:')[1].split('FEEDBACK:')[0].strip()
                    feedback = output.split('FEEDBACK:')[1].strip()

                    # TASK 1: CORRECTION TASK
                    correction_data.append({
                        'text': f"Correct the grammar errors in this sentence: {item['input']}",
                        'target': correction
                    })

                    # TASK 2: FEEDBACK TASK - Strong instruction for BART
                    feedback_data.append({
                        'text': f"Identify the specific grammar errors in this sentence and explain the correct grammar rules: {item['input']}",
                        'target': feedback
                    })

    return correction_data, feedback_data

# Load data
correction_data, feedback_data = load_data_bart_dual_task(jsonl_file)

# Combine both tasks for training
combined_data = correction_data + feedback_data
train_data, val_data = train_test_split(combined_data, test_size=0.1, random_state=42)

dataset = DatasetDict({
    'train': Dataset.from_list(train_data),
    'validation': Dataset.from_list(val_data)
})


In [7]:
# BART tokenization
def preprocess_function(examples):
    inputs = examples['text']
    targets = examples['target']

    # Tokenize inputs
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding=False)

    # Tokenize targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=512, truncation=True, padding=False)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Tokenize dataset
tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)
print("Dataset tokenized successfully!")

# Check a sample
sample = tokenized_dataset['train'][0]
print(f"Sample input length: {len(sample['input_ids'])}")
print(f"Sample labels length: {len(sample['labels'])}")


Map:   0%|          | 0/4237 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/471 [00:00<?, ? examples/s]

Dataset tokenized successfully!
Sample input length: 42
Sample labels length: 134


In [8]:
# BART data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    return_tensors="pt"
)

print("Data collator created!")


Data collator created!


In [9]:
# BART training arguments
training_args = TrainingArguments(
    output_dir="./bart-grammar-model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    warmup_steps=150,
    weight_decay=0.01,
    learning_rate=3e-5,  # BART works well with this learning rate
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=75,
    save_strategy="no",  # Disable checkpoint saving to save space
    save_steps=500,  # Not used since save_strategy="no"
    load_best_model_at_end=False,  # Disabled since no saving
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    report_to=None,
    max_grad_norm=1.0,
)

print("BART training arguments set!")
print(f"Training for {training_args.num_train_epochs} epochs")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")


BART training arguments set!
Training for 3 epochs
Batch size: 2
Learning rate: 3e-05


In [10]:
# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("BART trainer initialized successfully!")


BART trainer initialized successfully!


/tmp/ipython-input-1980118804.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [11]:
# Start training
print("Starting BART-LARGE training...")
trainer.train()
print("Training completed!")


Starting BART-LARGE training...


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: e-obolo (j-chemirmir-glasgow-caledonian-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss
75,2.143800,1.332573
150,1.286800,1.194164
225,1.093200,1.187264
300,1.188000,1.096009
375,1.035600,1.138386
450,1.296000,1.002841
525,1.062200,1.015481
600,1.065700,0.958149
675,1.066000,1.010028
750,1.041400,0.985424


Training completed!


In [12]:
# Test the FINE-TUNED model with dual task approach
print("=== TESTING FINE-TUNED MODEL - DUAL TASK APPROACH ===")

def test_correction_task(text):
    input_text = f"Correct the grammar errors in this sentence: {text}"
    inputs = tokenizer(input_text, return_tensors="pt", max_length=1024, truncation=True)

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=500,
            num_beams=4,
            early_stopping=True,
            do_sample=False
        )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result

def test_feedback_task(text):
    input_text = f"Explain the grammar error in this sentence: {text}"
    inputs = tokenizer(input_text, return_tensors="pt", max_length=1024, truncation=True)

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=500,
            num_beams=4,
            early_stopping=True,
            do_sample=False
        )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result

# Test sentences
test_sentences = [
    "I has a apple.",
    "She go to school yesterday.",
    "The students was very happy.",
    "He don't like apples.",
    "We was playing football."
]

print("=== TASK 1: GRAMMAR CORRECTION TEST ===")
for sentence in test_sentences:
    print(f"\nInput: {sentence}")
    correction = test_correction_task(sentence)
    print(f"Correction: {correction}")
    print("-" * 50)

print("\n=== TASK 2: FEEDBACK GENERATION TEST ===")
for sentence in test_sentences:
    print(f"\nInput: {sentence}")
    feedback = test_feedback_task(sentence)
    print(f"Feedback: {feedback}")
    print("-" * 50)

print("\n=== COMBINED OUTPUT ===")
for sentence in test_sentences:
    correction = test_correction_task(sentence)
    feedback = test_feedback_task(sentence)
    print(f"\nInput: {sentence}")
    print(f"CORRECTION: {correction}")
    print(f"FEEDBACK: {feedback}")
    print("-" * 50)

=== TESTING FINE-TUNED MODEL - DUAL TASK APPROACH ===
=== TASK 1: GRAMMAR CORRECTION TEST ===

Input: I has a apple.
Correction: I have an apple.
--------------------------------------------------

Input: She go to school yesterday.
Correction: She went to school yesterday.
--------------------------------------------------

Input: The students was very happy.
Correction: The students were very happy.
--------------------------------------------------

Input: He don't like apples.
Correction: He doesn't like apples.
--------------------------------------------------

Input: We was playing football.
Correction: We were playing football.
--------------------------------------------------

=== TASK 2: FEEDBACK GENERATION TEST ===

Input: I has a apple.
Feedback: I have an apple.
--------------------------------------------------

Input: She go to school yesterday.
Feedback: She went to school yesterday. This sentence is grammatically correct. It uses proper phrasing and tense to describe 

In [13]:
# Evaluate the fine-tuned model using BLEU and ROUGE
print("=== EVALUATING FINE-TUNED MODEL ===")

# Function to generate predictions for evaluation
def generate_predictions(dataset, model, tokenizer, device, batch_size=8):
    predictions = []
    references = []
    model.eval()
    for i in range(0, len(dataset), batch_size):
        batch = dataset[i : i + batch_size]
        inputs = tokenizer(batch['text'], return_tensors="pt", max_length=1024, truncation=True, padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=512,
                num_beams=4,
                early_stopping=True,
                do_sample=False
            )

        decoded_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend(decoded_preds)
        references.extend(batch['target']) # Use the original target from the dataset

    return predictions, references

# Generate predictions and references for the validation set
val_predictions, val_references = generate_predictions(dataset["validation"], model, tokenizer, device)

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

# Compute BLEU score
bleu_results = bleu.compute(predictions=val_predictions, references=val_references)
print(f"BLEU score: {bleu_results['bleu']:.4f}")

# Compute ROUGE scores
# ROUGE expects a list of lists for references when there are multiple possible references per prediction
# In this case, we only have one reference per prediction, so we wrap each reference in a list.
rouge_references = [[ref] for ref in val_references]
rouge_results = rouge.compute(predictions=val_predictions, references=rouge_references)
print(f"ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_results['rouge2']:.4f}")
print(f"ROUGE-L: {rouge_results['rougeL']:.4f}")

print("Evaluation completed!")

=== EVALUATING FINE-TUNED MODEL ===


BLEU score: 0.3799
ROUGE-1: 0.7174
ROUGE-2: 0.5415
ROUGE-L: 0.6748
Evaluation completed!
